# Fire2Air Darwin — Assessment 3
## 02 Model 1 Classification: Next-Day PM2.5 Exceedance Risk - AWS Notebook
**Member:** Lihini Jinanjalie S386228

In [2]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    auc, brier_score_loss, ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print("XGBoost unavailable:", exc)

AWS Configuration

In [ ]:
import boto3
import shutil

BUCKET = "fire2air-darwin-dan5-646468992009-ap-southeast-2-an"

NTEPA_RAW = f"s3://{BUCKET}/raw/ntepa/"
FIRMS_RAW = f"s3://{BUCKET}/raw/firms/"
NTEPA_DAILY = f"s3://{BUCKET}/processed/ntepa_daily/"
FIRMS_DAILY = f"s3://{BUCKET}/processed/firms_daily/"
MODEL_READY = f"s3://{BUCKET}/processed/model_ready/"
MODEL1_DIR = f"s3://{BUCKET}/models/model1/"
MODEL2_DIR = f"s3://{BUCKET}/models/model2/"
MODEL3_DIR = f"s3://{BUCKET}/models/model3/"
MODEL1_OUTPUT = f"s3://{BUCKET}/outputs/model1/"
MODEL2_OUTPUT = f"s3://{BUCKET}/outputs/model2/"
MODEL3_OUTPUT = f"s3://{BUCKET}/outputs/model3/"
VIS_OUTPUT = f"s3://{BUCKET}/outputs/visualisation/"
DASHBOARD_DIR = f"s3://{BUCKET}/dashboard/"
DOCUMENTATION_DIR = f"s3://{BUCKET}/documentation/"
ATHENA_RESULTS = f"s3://{BUCKET}/athena-results/"

s3 = boto3.client("s3")
LOCAL_ROOT = Path("/tmp/fire2air_a3")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def _prefix(uri):
    return uri.replace(f"s3://{BUCKET}/", "", 1)

def download_prefix(uri, local_dir, flatten=False):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    prefix = _prefix(uri)
    paginator = s3.get_paginator("list_objects_v2")
    downloaded = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            rel = Path(key[len(prefix):].lstrip("/"))
            dest = local_dir / (rel.name if flatten else rel)
            dest.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(BUCKET, key, str(dest))
            downloaded.append(dest)
    return downloaded

def upload_file(local_path, uri, object_name=None):
    local_path = Path(local_path)
    prefix = _prefix(uri).rstrip("/")
    key = f"{prefix}/{object_name or local_path.name}"
    s3.upload_file(str(local_path), BUCKET, key)
    print(f"Uploaded: s3://{BUCKET}/{key}")

def upload_tree(local_dir, uri):
    local_dir = Path(local_dir)
    if not local_dir.exists():
        return
    prefix = _prefix(uri).rstrip("/")
    for p in local_dir.rglob("*"):
        if p.is_file():
            rel = p.relative_to(local_dir).as_posix()
            key = f"{prefix}/{rel}"
            s3.upload_file(str(p), BUCKET, key)
    print(f"Uploaded folder: {local_dir} -> {uri}")

RANDOM_STATE = 42
BOOTSTRAP_REPEATS = 300

project_folder = LOCAL_ROOT / "model1_workspace"
if project_folder.exists():
    shutil.rmtree(project_folder)
project_folder.mkdir(parents=True, exist_ok=True)

# Download the shared model-ready checkpoint and manifest produced by Script 01.
download_prefix(MODEL_READY, project_folder, flatten=True)

output_root = project_folder / "outputs_prt661_a3"
table_dir = output_root / "tables"
figure_dir = output_root / "figures"
model_dir = output_root / "models"
for folder in [table_dir, figure_dir, model_dir]:
    folder.mkdir(parents=True, exist_ok=True)

integrated_daily = pd.read_csv(
    project_folder / "Fire2Air_model_ready_checkpoint_A3.csv",
    parse_dates=["date", "target_date"],
)
with open(project_folder / "Fire2Air_feature_manifest_A3.json", "r", encoding="utf-8") as f:
    manifest = json.load(f)
model_features = manifest["features"]
feature_groups = manifest["feature_groups"]
classification_target = manifest["targets"]["classification"]

print("Project folder:", project_folder)
print("Dataset shape:", integrated_daily.shape)
print("Assessment 3 predictors:", len(model_features))

## Chronological split and class balanc

In [ ]:
classification_data = integrated_daily.dropna(
    subset=[classification_target, "target_date"]
).copy()
classification_data[classification_target] = classification_data[classification_target].astype(int)
classification_data["target_year"] = classification_data["target_date"].dt.year

train_cls = classification_data[classification_data["target_year"] <= 2022].copy()
validation_cls = classification_data[classification_data["target_year"] == 2023].copy()
test_cls = classification_data[classification_data["target_year"] == 2024].copy()

if min(len(train_cls), len(validation_cls), len(test_cls)) == 0:
    raise ValueError(
        f"Insufficient chronological data: train={len(train_cls)}, "
        f"validation={len(validation_cls)}, test={len(test_cls)}"
    )
assert train_cls["target_date"].max() < validation_cls["target_date"].min()
assert validation_cls["target_date"].max() < test_cls["target_date"].min()

balance_rows = []
for split_name, data in [("Train", train_cls), ("Validation", validation_cls), ("Test", test_cls)]:
    positives = int(data[classification_target].sum())
    balance_rows.append({
        "Split": split_name,
        "Rows": len(data),
        "Positive_Days": positives,
        "Negative_Days": int(len(data) - positives),
        "Positive_Rate": positives / len(data),
    })
class_balance = pd.DataFrame(balance_rows)
class_balance.to_csv(table_dir / "a3_model1_class_balance.csv", index=False)
print("\nClass balance:")
print(class_balance.round(4).to_string(index=False))

X_train, y_train = train_cls[model_features], train_cls[classification_target]
X_validation, y_validation = validation_cls[model_features], validation_cls[classification_target]
X_test, y_test = test_cls[model_features], test_cls[classification_target]

## Reusable preprocessing/model builders

In [ ]:
def onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(features, scale_numeric=False):
    categorical = [f for f in features if f == "station"]
    numeric = [f for f in features if f not in categorical]

    transformers = []
    if categorical:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", onehot()),
            ]),
            categorical,
        ))
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    if numeric:
        transformers.append(("numeric", Pipeline(numeric_steps), numeric))
    return ColumnTransformer(transformers)


def build_classifier(model_name, params, features, scale_pos_weight):
    if model_name == "Logistic Regression":
        estimator = LogisticRegression(
            C=params["C"],
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )
        prep = build_preprocessor(features, scale_numeric=True)
    elif model_name == "Random Forest":
        estimator = RandomForestClassifier(
            n_estimators=300,
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        prep = build_preprocessor(features, scale_numeric=False)
    elif model_name == "XGBoost":
        if not HAS_XGBOOST:
            raise RuntimeError("XGBoost was selected but the package is unavailable.")
        estimator = XGBClassifier(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        prep = build_preprocessor(features, scale_numeric=False)
    else:
        raise ValueError(f"Unknown classifier: {model_name}")
    return Pipeline([("preprocessor", prep), ("model", estimator)])


def safe_probability_metrics(y_true, probabilities):
    y_arr = np.asarray(y_true)
    p_arr = np.asarray(probabilities)
    out = {"PR_AUC": np.nan, "Average_Precision": np.nan, "ROC_AUC": np.nan, "Brier": np.nan}
    if len(np.unique(y_arr)) < 2:
        return out
    precision_curve, recall_curve, _ = precision_recall_curve(y_arr, p_arr)
    out["PR_AUC"] = auc(recall_curve, precision_curve)
    out["Average_Precision"] = average_precision_score(y_arr, p_arr)
    out["ROC_AUC"] = roc_auc_score(y_arr, p_arr)
    out["Brier"] = brier_score_loss(y_arr, p_arr)
    return out


def threshold_metrics(y_true, probabilities, threshold):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "False_Negatives": int(((np.asarray(y_true) == 1) & (predictions == 0)).sum()),
        "False_Positives": int(((np.asarray(y_true) == 0) & (predictions == 1)).sum()),
    }


def find_best_threshold(y_true, probabilities):
    thresholds = np.arange(0.01, 1.00, 0.01)
    scores = [
        f1_score(y_true, (np.asarray(probabilities) >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    best_idx = int(np.argmax(scores))
    return float(thresholds[best_idx]), float(scores[best_idx])

## Candidate models and PR-AUC-first tuning

In [ ]:
pos = int(y_train.sum())
neg = int(len(y_train) - pos)
scale_pos_weight = neg / max(pos, 1)

candidate_specs = []
for c_value in [0.01, 0.1, 1.0, 10.0]:
    candidate_specs.append(("Logistic Regression", {"C": c_value}, f"C={c_value}"))
for max_depth, min_samples_leaf in [(None, 1), (None, 4), (10, 4), (15, 4)]:
    candidate_specs.append((
        "Random Forest",
        {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf},
        f"max_depth={max_depth};min_samples_leaf={min_samples_leaf}",
    ))
if HAS_XGBOOST:
    for n_estimators, learning_rate, max_depth in [
        (100, 0.05, 2), (200, 0.05, 2), (200, 0.05, 3), (300, 0.05, 3)
    ]:
        candidate_specs.append((
            "XGBoost",
            {"n_estimators": n_estimators, "learning_rate": learning_rate, "max_depth": max_depth},
            f"n_estimators={n_estimators};learning_rate={learning_rate};max_depth={max_depth}",
        ))

validation_rows = []
trained_templates = {}
params_lookup = {}
for model_family, params, config_id in candidate_specs:
    pipeline = build_classifier(model_family, params, model_features, scale_pos_weight)
    pipeline.fit(X_train, y_train)
    probabilities = pipeline.predict_proba(X_validation)[:, 1]
    threshold, _ = find_best_threshold(y_validation, probabilities)
    row = {
        "Model": model_family,
        "Config": config_id,
        "Threshold": threshold,
        **safe_probability_metrics(y_validation, probabilities),
        **threshold_metrics(y_validation, probabilities, threshold),
    }
    validation_rows.append(row)
    trained_templates[(model_family, config_id)] = pipeline
    params_lookup[(model_family, config_id)] = params

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["PR_AUC", "F1"], ascending=[False, False]
).reset_index(drop=True)
validation_results.to_csv(table_dir / "a3_model1_classification_validation_results.csv", index=False)
print("\n2023 validation configurations:")
print(validation_results.round(4).to_string(index=False))

best_by_family = (
    validation_results.sort_values(["PR_AUC", "F1"], ascending=[False, False])
    .groupby("Model", as_index=False).first()
    .sort_values(["PR_AUC", "F1"], ascending=[False, False])
    .reset_index(drop=True)
)
best_by_family.to_csv(table_dir / "a3_model1_best_by_family.csv", index=False)
selected = best_by_family.iloc[0]
best_model_name = selected["Model"]
best_config_id = selected["Config"]
final_threshold = float(selected["Threshold"])
selected_params = params_lookup[(best_model_name, best_config_id)]
selected_template = trained_templates[(best_model_name, best_config_id)]
print("\nSelected model by validation PR-AUC:", best_model_name)
print("Configuration:", best_config_id)
print("Validation F1 threshold:", round(final_threshold, 3))